In [ ]:
# # 1. UPDATED MOUSE CLICK FUNCTION WITH SAFETY LIMIT
clicked_points = []
def click_event(event, x, y, flags, params):
    global clicked_points
    if event == cv2.EVENT_LBUTTONDOWN:
        # Stop registering clicks if we already have 4 points
        if len(clicked_points) < 4:
            clicked_points.append([x, y])
            cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
            
            # Print feedback to terminal so you can keep track
            print(f"Point {len(clicked_points)} registered at ({x}, {y})")
            
            # Update the display window
            title = "Click 4 inner corners: Top-Left, Top-Right, Bottom-Left, Bottom-Right"
            cv2.imshow(title, img_copy)
        else:
            print("Warning: Already registered 4 points! Extra click ignored.")

In [ ]:
import numpy as np
import cv2
import glob
import os

# 1. SETUP TERMINATION CRITERIA FOR SUB-PIXEL ACCURACY
# Stops when accuracy hits 0.001 or after 30 loops
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 2. DEFINE THE CHESSBOARD GEOMETRY
# Number of INSIDE corners on your physical calibration grid
CHECKERBOARD = (2, 2)  # Change this to match your specific board size
SQUARE_SIZE = 0.127    # Size of a square side in meters (e.g., 25mm = 0.025m)

# 3. PREPARE 3D OBJECT POINTS
# These are coordinates like (0,0,0), (1,0,0), (2,0,0) ...
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

# Arrays to store 3D points and 2D image points from all images
objpoints = []  # 3D points in real-world space
imgpoints_l = []  # 2D points in left camera image plane
imgpoints_r = []  # 2D points in right camera image plane

# 4. LOAD IMAGES
# Make sure your file names or paths match your image folders
base_path = os.path.expanduser('~/dev/reachy-2026-iitg/reachy-tabletop-ai/data/calibration/calib_images')

images_left = sorted(glob.glob(os.path.join(base_path, 'left', '*.png')))
images_right = sorted(glob.glob(os.path.join(base_path, 'right', '*.png')))

# Ensure we have a matching pairs of stereo images
assert len(images_left) == len(images_right), "Number of left and right images must match!"

# 5. FIND CORNERS IN BOTH CAMERAS

for img_l_path, img_r_path in zip(images_left, images_right):
    img_l = cv2.imread(img_l_path)
    img_r = cv2.imread(img_r_path)
    gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

    window_title = "Click 4 inner corners: Top-Left, Top-Right, Bottom-Left, Bottom-Right"

    # --- PROCESS LEFT IMAGE ---
    print(f"\n--- Processing Left Image: {os.path.basename(img_l_path)} ---")
    clicked_points = []
    img_copy = img_l.copy()
    cv2.imshow(window_title, img_copy)
    cv2.setMouseCallback(window_title, click_event)
    cv2.waitKey(0) # Press ANY KEY on your keyboard after making your 4 clicks
    
    # Check if exactly 4 points were collected
    if len(clicked_points) != 4:
        print(f"Error: You clicked {len(clicked_points)} times. Must be exactly 4. Skipping this pair.")
        continue
    corners_l = np.array(clicked_points, dtype=np.float32).reshape(-1, 1, 2)

    # --- PROCESS RIGHT IMAGE ---
    print(f"--- Processing Right Image: {os.path.basename(img_r_path)} ---")
    clicked_points = []
    img_copy = img_r.copy()
    cv2.imshow(window_title, img_copy)
    cv2.setMouseCallback(window_title, click_event)
    cv2.waitKey(0) # Press ANY KEY on your keyboard after making your 4 clicks
    
    # Check if exactly 4 points were collected
    if len(clicked_points) != 4:
        print(f"Error: You clicked {len(clicked_points)} times. Must be exactly 4. Skipping this pair.")
        continue
    corners_r = np.array(clicked_points, dtype=np.float32).reshape(-1, 1, 2)

    cv2.destroyAllWindows()

    # Refine the clicked points to sub-pixel accuracy automatically
    corners_l = cv2.cornerSubPix(gray_l, corners_l, (11, 11), (-1, -1), criteria)
    corners_r = cv2.cornerSubPix(gray_r, corners_r, (11, 11), (-1, -1), criteria)

    objpoints.append(objp)
    imgpoints_l.append(corners_l)
    imgpoints_r.append(corners_r)

# Get the resolution of the images (width, height)
img_shape = gray_l.shape[::-1]

# 6. STEP 1: INDIVIDUAL CAMERA CALIBRATION (INTRINSICS)
# Calibrate left camera
ret_l, mtx_l, dist_l, rvecs_l, tvecs_l = cv2.calibrateCamera(
    objpoints, imgpoints_l, img_shape, None, None
)

# Calibrate right camera
ret_r, mtx_r, dist_r, rvecs_r, tvecs_r = cv2.calibrateCamera(
    objpoints, imgpoints_r, img_shape, None, None
)

# 7. STEP 2: STEREO CALIBRATION (EXTRINSICS)
# This calculates the physical distance and rotation between the two cameras
flags = cv2.CALIB_FIX_INTRINSIC  # Keep individual camera shapes locked

retval, mtx_l, dist_l, mtx_r, dist_r, R, T, E, F = cv2.stereoCalibrate(
    objpoints, imgpoints_l, imgpoints_r,
    mtx_l, dist_l, mtx_r, dist_r,
    img_shape, criteria=criteria, flags=flags
)

# 8. STEP 3: RECTIFICATION FOR DEPTH MAPPING
# This creates alignment maps so pixels match up horizontally for stereo depth matching
R1, R2, P1, P2, Q, validRoi1, validRoi2 = cv2.stereoRectify(
    mtx_l, dist_l, mtx_r, dist_r, img_shape, R, T
)

# 9. SAVE THE CALIBRATION PARAMS TO DISK
# You can load this .npz file later into your real-time depth matching code
np.savez('stereo_calibration.npz', 
         mtx_l=mtx_l, dist_l=dist_l, 
         mtx_r=mtx_r, dist_r=dist_r, 
         R=R, T=T, R1=R1, R2=R2, P1=P1, P2=P2, Q=Q)

print("Stereo Calibration Complete! File saved as 'stereo_calibration.npz'")
print(f"Cameras are spaced apart by (Translation Vector T in meters):\n {T}")


In [ ]:
import numpy as np
from datetime import datetime

# Load your completed calibration file
data = np.load('stereo_calibration.npz')

# Extract left and right intrinsic matrices
mtx_l = data['mtx_l']
mtx_r = data['mtx_r']

# Extract distortion arrays and ensure they are flat lists
dist_l = data['dist_l'].flatten().tolist()
dist_r = data['dist_r'].flatten().tolist()

# Extract structural Extrinsics
R_mat = data['R'].tolist()
T_vec = data['T'].flatten().tolist()

# Baseline calculation (Physical distance between cameras)
# The absolute value of the X-axis coordinate in the translation vector is the baseline
baseline_m = abs(T_vec[0]) 

# Print out your formatted values
print("--- COPY AND PASTE THESE VALUES INTO YOUR JSON ---")
print(f'  "baseline_m": {baseline_m},')
print(f'  "image_width": 1920,')
print(f'  "image_height": 1080,')
print('  "camera_left": {')
print(f'    "fx": {mtx_l[0,0]}, "fy": {mtx_l[1,1]}, "cx": {mtx_l[0,2]}, "cy": {mtx_l[1,2]},')
print(f'    "distortion": {dist_l[:5]}')
print('  },')
print('  "camera_right": {')
print(f'    "fx": {mtx_r[0,0]}, "fy": {mtx_r[1,1]}, "cx": {mtx_r[0,2]}, "cy": {mtx_r[1,2]},')
print(f'    "distortion": {dist_r[:5]}')
print('  },')
print(f'  "R": {R_mat},')
print(f'  "T": {T_vec},')
print(f'  "calibrated_at": "{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}",')
print('  "rms_error_px": null')
